# Run the experiment in batches

Run the cells **one at a time, from top to bottom**. Click the play button (▶) at the left of a cell, wait until it finishes, then move to the next.

- **Step 1:** setup (once per session).
- **Step 2:** settings and secrets. Secrets are typed into hidden prompts and never stored in the notebook.
- **Step 3:** run one batch (this takes a while; keep the tab open).
- **Step 4:** see the results of the batch and download them.
- **Step 5:** only after the last batch: merge all batches and make the final statistics.
- **Step 6:** delete the saved secrets.

If Colab disconnects, run Step 1 and Step 2 again. Batch files you already downloaded are safe on your computer.

In [ ]:
#@title Step 1: Setup (run once). Wait for SETUP DONE.
import os, re, subprocess, zipfile
from google.colab import files

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)

# Node.js 20.6 or newer
ver = sh("node --version").stdout.strip()
nums = tuple(int(x) for x in re.findall(r"\d+", ver)[:2]) if ver else (0, 0)
if nums < (20, 6):
    sh("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs")
print("Node:", sh("node --version").stdout.strip())

os.chdir("/content")
print("1/2  Choose blockchain-ipfs-storage.zip")
up = files.upload()
with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall("/content")
os.chdir("/content/blockchain-ipfs-storage")

print("2/2  Choose your four sample files: sample_100KB.txt, sample_1MB.txt, sample_10MB.txt, sample_50MB.txt")
os.makedirs("data", exist_ok=True)
up = files.upload()
for name, content in up.items():
    with open(f"data/{name}", "wb") as f:
        f.write(content)

print(sh("npm install --no-audit --no-fund").stdout[-300:])
sh("pip install -q numpy scipy pandas matplotlib")
r = sh("npm run samples", env=dict(os.environ, SAMPLE_SIZES="512KiB,5MiB,25MiB"))
print(r.stdout[-400:])

need = ["data/sample_100KB.txt", "data/sample_512KiB.bin", "data/sample_1MB.txt", "data/sample_5MiB.bin",
        "data/sample_10MB.txt", "data/sample_25MiB.bin", "data/sample_50MB.txt"]
missing = [f for f in need if not os.path.exists(f)]
if missing:
    print("MISSING FILES:", missing)
else:
    print("SETUP DONE")

In [ ]:
#@title Step 2: Settings and secrets. Fill the boxes at the right, then run.
BATCH = 1  #@param {type:"integer"}
RUNS_PER_FILE = 10  #@param {type:"integer"}
CONTRACT_ADDRESS = ""  #@param {type:"string"}

import getpass, os, re, subprocess

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)

os.chdir("/content/blockchain-ipfs-storage")
FILES = ["data/sample_100KB.txt", "data/sample_512KiB.bin", "data/sample_1MB.txt", "data/sample_5MiB.bin",
         "data/sample_10MB.txt", "data/sample_25MiB.bin", "data/sample_50MB.txt"]
OUT = f"results/batch{BATCH}.json"

pinata_jwt = getpass.getpass("Pinata JWT: ")
rpc_url = getpass.getpass("Alchemy Sepolia URL: ")
private_key = getpass.getpass("Private key of the test account: ")
if not private_key.startswith("0x"):
    private_key = "0x" + private_key

lines = [f"PINATA_JWT={pinata_jwt}", f"RPC_URL={rpc_url}", f"PRIVATE_KEY={private_key}",
         f"FILES={','.join(FILES)}", f"RUNS={RUNS_PER_FILE}", "CONFIRMATIONS=1",
         "GATEWAY=https://gateway.pinata.cloud/ipfs", f"OUT={OUT}"]
with open(".env", "w") as f:
    f.write("\n".join(lines) + "\n")
os.chmod(".env", 0o600)
del pinata_jwt, rpc_url, private_key

# Contract: use the address you typed, or deploy a new one if the box is empty
address = CONTRACT_ADDRESS.strip()
if not address:
    r = sh("npx hardhat run scripts/deploy.js --network sepolia")
    print(r.stdout, r.stderr[-1200:])
    m = re.search(r"deployed to:\s*(0x[0-9a-fA-F]{40})", r.stdout)
    assert m, "Deployment failed: see the output above"
    address = m.group(1)
    print("SAVE THIS ADDRESS:", address)
with open(".env", "a") as f:
    f.write(f"CONTRACT={address}\n")

# Balance and cost check
check = """const { ethers } = require("ethers");
(async () => {
  const p = new ethers.providers.JsonRpcProvider(process.env.RPC_URL);
  const w = new ethers.Wallet(process.env.PRIVATE_KEY, p);
  const bal = await p.getBalance(w.address);
  const gp = await p.getGasPrice();
  const txs = Number(process.env.RUNS) * process.env.FILES.split(",").length;
  const cost = gp.mul(475000).mul(txs).mul(13).div(10);
  console.log("wallet:", w.address);
  console.log("balance (ETH):", ethers.utils.formatEther(bal));
  console.log("gas price (gwei):", ethers.utils.formatUnits(gp, "gwei"));
  console.log("this batch:", txs, "transactions, estimated cost with margin (ETH):", ethers.utils.formatEther(cost));
})();
"""
with open("scripts/check.js", "w") as f:
    f.write(check)
r = sh("node --env-file=.env scripts/check.js")
print(r.stdout, r.stderr[-800:])
print("CONTRACT =", address, "| batch", BATCH, "| runs per file", RUNS_PER_FILE)

In [ ]:
#@title Step 3: Run the batch (keep this tab open until it says done)
import os
os.chdir("/content/blockchain-ipfs-storage")
!node --env-file=.env scripts/measure.js

In [ ]:
#@title Step 4: Look at this batch and download it
import os, subprocess
from google.colab import files

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)

os.chdir("/content/blockchain-ipfs-storage")
out = f"results/batch{BATCH}.json"
r = sh(f"python analysis/stats.py {out} results/figs_batch{BATCH}")
print(r.stdout, r.stderr[-800:])
files.download(out)

In [ ]:
#@title Step 5: After the LAST batch: merge everything and make the final statistics
UPLOAD_EARLIER_BATCHES = False  #@param {type:"boolean"}
MIN_BATCH = 2  #@param {type:"integer"}

import glob, json, os, shutil, subprocess
import pandas as pd
from IPython.display import Image, display
from google.colab import files

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)

os.chdir("/content/blockchain-ipfs-storage")
if UPLOAD_EARLIER_BATCHES:   # tick the box if Colab was reset and you need to re-upload your batch files
    up = files.upload()
    for name, content in up.items():
        with open(f"results/{name}", "wb") as f:
            f.write(content)

rows, failures, meta = [], [], None
paths = sorted(glob.glob("results/batch*.json"))
for path in paths:
    d = json.load(open(path))
    n = int(path.split("batch")[-1].split(".")[0])
    if n < MIN_BATCH:   # batch 1 was a pilot with the old timing settings
        continue
    meta = meta or d["meta"]
    rows += [dict(r, batch=n) for r in d["rows"]]
    failures += [dict(f, batch=n) for f in d["failures"]]
with open("results/results.json", "w") as f:
    json.dump({"meta": meta, "rows": rows, "failures": failures}, f, indent=2)
print(len(rows), "rows from batch files numbered", MIN_BATCH, "and above")

r = sh("python analysis/stats.py results/results.json results/figs")
print(r.stdout, r.stderr[-800:])

summary = pd.read_csv("results/figs/summary.csv")
display(summary[summary.measure.isin(["total", "confirm", "gasUsed"])].round(3))
for name in ["fig_stages.png", "fig_total.png", "fig_onchain.png"]:
    display(Image(f"results/figs/{name}"))

env = {"node": sh("node --version").stdout.strip(), "hardhat": sh("npx hardhat --version").stdout.strip(),
       "contract": meta["contract"] if meta else None}
with open("results/environment.json", "w") as f:
    json.dump(env, f, indent=2)
shutil.make_archive("/content/results_bundle", "zip", "results")
files.download("/content/results_bundle.zip")

In [ ]:
#@title Step 6: Delete the saved secrets (run when you are finished for now)
import os
if os.path.exists("/content/blockchain-ipfs-storage/.env"):
    os.remove("/content/blockchain-ipfs-storage/.env")
print("Secrets file removed. Revoke the Pinata key when the whole experiment is done.")